# BMW AI — Module 1D: Train YOLOv8n Driver Monitoring (Colab)

Fine-tune **YOLOv8n** for cabin object detection:
- `0` = phone
- `1` = smoking
- `2` = no_seatbelt

**Runtime:** Runtime → Change runtime type → **GPU** (T4)

**Dataset:** Official [Vicomtech DMD](https://github.com/Vicomtech/DMD-Driver-Monitoring-Dataset) is **not** on Kaggle/Colab by default.  
Request access → convert to YOLO format → upload a zip (or put it on Google Drive).

Expected layout after unzip:
```
dataset/
  images/train/
  images/val/
  labels/train/
  labels/val/
  dataset.yaml   (optional — we can generate it)
```

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics==8.2.0

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Google Drive (recommended)

Saves checkpoints if the Colab session disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/bmw_ai")
DATA_DIR = DRIVE_ROOT / "datasets" / "dmd_yolo"
MODEL_DIR = DRIVE_ROOT / "models"
RUNS_DIR = DRIVE_ROOT / "runs"

for p in (DATA_DIR, MODEL_DIR, RUNS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("RUNS_DIR :", RUNS_DIR)

## 3. Get the dataset into Colab

Pick **one** option below.

### Option A — Upload a ZIP from your PC

Zip your YOLO folder so the archive contains `images/`, `labels/` (and optionally `dataset.yaml`).

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

LOCAL_DATA = Path("/content/dmd_yolo")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # choose your .zip

for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name, "r") as zf:
            zf.extractall(LOCAL_DATA)
        print("Extracted", name, "->", LOCAL_DATA)

# If the zip nested an extra folder, find images/train
candidates = list(LOCAL_DATA.rglob("images/train"))
if candidates:
    DATASET_ROOT = candidates[0].parent.parent
else:
    DATASET_ROOT = LOCAL_DATA
print("DATASET_ROOT:", DATASET_ROOT)

### Option B — Already on Google Drive

Put the YOLO dataset under `MyDrive/bmw_ai/datasets/dmd_yolo/` then run:

In [ ]:
from pathlib import Path

# Uncomment if you skipped Option A:
# DATASET_ROOT = Path("/content/drive/MyDrive/bmw_ai/datasets/dmd_yolo")

assert "DATASET_ROOT" in dir() or "DATASET_ROOT" in globals(), "Set DATASET_ROOT first (Option A or B)"
DATASET_ROOT = Path(DATASET_ROOT)
assert (DATASET_ROOT / "images" / "train").is_dir(), f"Missing images/train under {DATASET_ROOT}"
assert (DATASET_ROOT / "labels" / "train").is_dir(), f"Missing labels/train under {DATASET_ROOT}"
print("OK:", DATASET_ROOT)

## 4. Write `dataset.yaml`

In [ ]:
from pathlib import Path

DATASET_ROOT = Path(DATASET_ROOT)
YAML_PATH = DATASET_ROOT / "dataset.yaml"

yaml_text = f"""path: {DATASET_ROOT.resolve().as_posix()}
train: images/train
val: images/val

nc: 3
names:
  0: phone
  1: smoking
  2: no_seatbelt
"""
YAML_PATH.write_text(yaml_text, encoding="utf-8")
print(YAML_PATH.read_text())

n_train = len(list((DATASET_ROOT / "images" / "train").glob("*.*")))
n_val = len(list((DATASET_ROOT / "images" / "val").glob("*.*")))
print(f"train images: {n_train} | val images: {n_val}")
assert n_train > 0, "No training images found"

## 5. Train YOLOv8n

~2–5 hours on Colab T4 for 100 epochs (reduce `EPOCHS` for a smoke test).

In [ ]:
from pathlib import Path
from ultralytics import YOLO

EPOCHS = 100          # use 5–10 for a quick smoke test
BATCH = 16            # lower to 8 if you hit CUDA OOM
IMGSZ = 640
NAME = "driver_monitor_dmd_v1"

# Prefer Drive for runs if mounted; else local /content
try:
    PROJECT = str(RUNS_DIR)
except NameError:
    PROJECT = "/content/runs"

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=15,
    save_period=10,
    project=PROJECT,
    name=NAME,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

metrics = results.results_dict
print(f"Best mAP50: {metrics.get('metrics/mAP50(B)', 'n/a')}")
print(f"Best mAP50-95: {metrics.get('metrics/mAP50-95(B)', 'n/a')}")
print("save_dir:", results.save_dir)

## 6. Copy `best.pt` to Drive + download locally

On your PC, save the file as:
`F:/BMW/ml/models/driver_monitor_best.pt`

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

best = Path(results.save_dir) / "weights" / "best.pt"
assert best.is_file(), f"best.pt not found at {best}"

try:
    dest = MODEL_DIR / "driver_monitor_best.pt"
    shutil.copy2(best, dest)
    print("Copied to Drive:", dest)
except NameError:
    dest = Path("/content/driver_monitor_best.pt")
    shutil.copy2(best, dest)
    print("Copied to:", dest)

files.download(str(dest))
print("Download started — place file at ml/models/driver_monitor_best.pt in the BMW repo")

## 7. Quick inference smoke test (optional)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

weights = Path(dest)
model = YOLO(str(weights))

# Pick any training image for a sanity check
sample_images = list((DATASET_ROOT / "images" / "val").glob("*.*"))
if not sample_images:
    sample_images = list((DATASET_ROOT / "images" / "train").glob("*.*"))

if sample_images:
    pred = model.predict(str(sample_images[0]), conf=0.25, verbose=False)
    print("Sample:", sample_images[0].name)
    print("Detections:", len(pred[0].boxes) if pred[0].boxes is not None else 0)
else:
    print("No sample images found")